## Uploading Vector Embeddings to Azure Cosmos DB

### Installing Libraries and Utilities

In [ ]:
%pip install azure-cosmos==4.16.0 azure-identity python-dotenv openai==2.38.0

### Setting up the Environment

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

# fetching the cosmosdb configuration from environment variables
cosmosdb_endpoint = os.getenv("COSMOSDB_ENDPOINT")
cosmosdb_key = os.getenv("COSMOSDB_KEY")
database_name = os.getenv("DATABASE_NAME")
container_name = os.getenv("CONTAINER_NAME") + "Vector"

# fetching the azure openai configuration from environment variables
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Creating the CosmosDB Client

In [ ]:
from azure.cosmos import CosmosClient
from azure.cosmos import PartitionKey

client = CosmosClient(cosmosdb_endpoint, cosmosdb_key)

### Navigate the Resource Hierarchy

In [ ]:
database = client.get_database_client(database_name)

### Create a Vector Embedding and Indexing Policy in a New Container

In [ ]:
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path":"/vector",
            "dataType":"float32",
            "distanceFunction":"cosine",
            "dimensions":1536
        }
    ]
}

vector_indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [
        {
            "path": "/*"
        }
    ],
    "excludedPaths": [
        {
            "path": "/_etag/?"
        },
        {
            "path": "/vector/*"
        }
    ],
    "vectorIndexes": [
        {
        "path": "/vector",
        "type": "diskANN"
        }
    ],
    "fullTextIndexes": []
}

container = database.create_container_if_not_exists(
    id = container_name,
    partition_key = PartitionKey(path="/category"),
    indexing_policy = vector_indexing_policy,
    vector_embedding_policy = vector_embedding_policy 
)

print("Container with vector indexing and embedding policies is ready.")

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Creating the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(client, text):
    
    response = client.embeddings.create(
        input=text,
        model = embedding_model_name
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']
    

### Generate Vector Embeddings for the Food Dataset and Upsert into CosmosDB

In [ ]:
import json

file_path = "./Data/food_dataset.json"

with open(file_path) as f:
    data = json.load(f)
    

for obj in data:
    vector_embeddings = generate_embeddings(azure_openai_client, obj['content'])
    obj['vector'] = vector_embeddings
    container.upsert_item(obj)